# Module 1: Tập lồi và hàm lồi — code

Notebook đi kèm site *Tối ưu hóa phi tuyến & lồi*. Chạy tuần tự từng ô (Colab có sẵn numpy, scipy, sympy, cvxpy). Tự giải tay trước rồi đối chiếu kết quả.

## Hessian và dấu ma trận bằng sympy (dạng A, B)

f = 2x² − xy + y² + e^(x+y): Hessian và các minor dẫn đầu.

In [1]:
import sympy as sp
x, y = sp.symbols("x y", real=True)
f = 2*x**2 - x*y + y**2 + sp.exp(x + y)
H = sp.hessian(f, (x, y))
print("H =", H)
print("H11 =", H[0, 0], "| det =", sp.simplify(H.det()))

H = Matrix([[exp(x + y) + 4, exp(x + y) - 1], [exp(x + y) - 1, exp(x + y) + 2]])
H11 = exp(x + y) + 4 | det = 8*exp(x + y) + 7


## Kiểm tra lồi bằng giá trị riêng

Phân loại nhanh một ma trận đối xứng.

In [2]:
import numpy as np
def kind(A):
    ev = np.linalg.eigvalsh(np.array(A, float))
    if all(ev > 1e-9): return "xác định dương"
    if all(ev >= -1e-9): return "nửa xác định dương"
    if all(ev < -1e-9): return "xác định âm"
    return "không xác định"
for A in ([[2, 1], [1, 3]], [[1, 1], [1, 1]], [[1, 2], [2, 1]], [[-2, 1], [1, -2]]):
    print(A, "->", kind(A))

[[2, 1], [1, 3]] -> xác định dương
[[1, 1], [1, 1]] -> nửa xác định dương
[[1, 2], [2, 1]] -> không xác định
[[-2, 1], [1, -2]] -> xác định âm


## Đỉnh của đa diện (dạng D)

Thử mọi cặp ràng buộc: giao điểm, rồi kiểm khả thi.

In [3]:
import numpy as np, itertools
A = np.array([[-1, 0], [0, -1], [1, 1], [1, 0]], float); b = np.array([0, 0, 4, 3], float)
for i, j in itertools.combinations(range(4), 2):
    M = A[[i, j]]
    if abs(np.linalg.det(M)) < 1e-9: continue
    p = np.linalg.solve(M, b[[i, j]])
    print((i + 1, j + 1), p, "khả thi" if np.all(A @ p <= b + 1e-9) else "loại")

(1, 2) [-0. -0.] khả thi
(1, 3) [-0.  4.] khả thi
(2, 3) [ 4. -0.] loại
(2, 4) [ 3. -0.] khả thi
(3, 4) [3. 1.] khả thi


## Bài code · Kiểm tra lồi bằng Jensen ngẫu nhiên

Đầu vào: một hàm f và số lần thử. Đầu ra: có tìm thấy phản ví dụ của $f(\theta x+(1-\theta)y)\le\theta f(x)+(1-\theta)f(y)$ không? (thử cho $x^2+xy+y^2$ và $xy$.)

In [4]:
import numpy as np
rng = np.random.default_rng(0)
def violates(f, n=20000):
    for _ in range(n):
        x, y, t = rng.normal(size=2), rng.normal(size=2), rng.random()
        if f(t*x + (1-t)*y) > t*f(x) + (1-t)*f(y) + 1e-9: return (x, y, t)
    return None
print("x^2+xy+y^2:", violates(lambda p: p[0]**2 + p[0]*p[1] + p[1]**2))
print("x*y       :", violates(lambda p: p[0]*p[1]) is not None)

x^2+xy+y^2: None
x*y       : True


## Bài code · Điểm có thuộc bao lồi? (LP)

Đầu vào: các đỉnh $(0,0),(3,0),(0,3)$, điểm $x$. Đầu ra: có $\lambda\ge0,\sum\lambda=1$ với $x=\sum\lambda_iv_i$ không?

In [5]:
import numpy as np
from scipy.optimize import linprog
V = np.array([[0, 0], [3, 0], [0, 3.]])
def in_hull(x):
    A = np.vstack([V.T, np.ones(3)]); b = np.r_[x, 1]
    r = linprog(np.zeros(3), A_eq=A, b_eq=b, bounds=(0, None)); return r.status == 0, r.x
print(in_hull([1, 1])); print(in_hull([2, 2])[0])

(True, array([0.33333333, 0.33333333, 0.33333333]))
False


## Bài code · Hình chiếu lên tập lồi và siêu phẳng tách

Đầu vào: tập $\{x_1^2+x_2^2\le1\}$, điểm $y=(2,2)$. Đầu ra: hình chiếu $p$, $a=y-p$, $b=a^\top p$.

In [6]:
import numpy as np, cvxpy as cp
y = np.array([2., 2]); x = cp.Variable(2)
cp.Problem(cp.Minimize(cp.sum_squares(x - y)), [cp.sum_squares(x) <= 1]).solve()
p = x.value; a = y - p; b = a @ p
print("p =", p.round(4), " a =", a.round(4), " b =", round(float(b), 4))
print("a'y - b =", round(float(a @ y - b), 4), "> 0 : y nằm phía bên kia siêu phẳng")

p = [0.7071 0.7071]  a = [1.2929 1.2929]  b = 1.8284
a'y - b = 3.3431 > 0 : y nằm phía bên kia siêu phẳng


## Bài code · Dấu ma trận: so với minor dẫn đầu

Đầu ra: với $\begin{pmatrix}0&0\\0&-1\end{pmatrix}$, minor dẫn đầu và giá trị riêng.

In [7]:
import numpy as np
A = np.array([[0, 0], [0, -1.]])
print("D1 =", A[0, 0], " D2 =", np.linalg.det(A), " eig =", np.linalg.eigvalsh(A))

D1 = 0.0  D2 = 0.0  eig = [-1.  0.]
